In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

## 1. 데이터 로드


In [ ]:
ROOT = Path('../../')
RAW_DIR = ROOT / 'data' / 'raw'

df_f = pd.read_csv(RAW_DIR / 'sports_gb_F.csv')
df_l = pd.read_csv(RAW_DIR / 'sports_gb_L.csv')
df_t = pd.read_csv(RAW_DIR / 'sports_gb_total.csv')

## 2. 날짜 파싱


In [ ]:
for col in ['DateReg', 'Date1Dep', 'Date1Spo']:
    df_t[col] = pd.to_datetime(df_t[col])

df_f['DateBet'] = pd.to_datetime(df_f['DateBet'])
df_l['DateBet'] = pd.to_datetime(df_l['DateBet'])

## 3. 기본 feature 생성

`sports_gb_total.csv`에서 직접적으로 가져올 수 있는 feature rename


In [ ]:
df = df_t[[
    'UserID', 'CountryID', 'Gender', 'BirthYear', 'DateReg', 'Date1Dep', 'Date1Spo',
    'StakeF', 'StakeL', 'StakeA',
    'WinF',   'WinL',   'WinA',
    'BetsF',  'BetsL',  'BetsA',
    'DaysF',  'DaysL',  'DaysA',
]].copy()

df = df.rename(columns={
    'UserID'    : 'user_id',
    'CountryID' : 'country_id',
    'Gender'    : 'gender',
    'BirthYear' : 'birth_year',
    'DateReg'   : 'reg_date',
    'Date1Dep'  : 'first_deposit',
    'Date1Spo'  : 'first_bet',
    'StakeF'    : 'fixed_bet_amount',
    'StakeL'    : 'live_bet_amount',
    'StakeA'    : 'total_bet_amount',
    'WinF'      : 'fixed_win_amount',
    'WinL'      : 'live_win_amount',
    'WinA'      : 'total_win_amount',
    'BetsF'     : 'fixed_bet_cnt',
    'BetsL'     : 'live_bet_cnt',
    'BetsA'     : 'total_bet_cnt',
    'DaysF'     : 'fixed_active_days',
    'DaysL'     : 'live_active_days',
    'DaysA'     : 'total_active_days',
})

df.head()

In [ ]:
df.isnull().sum()

## 4. age_group 계산

기준년도: 2006년
10대:0 / 20대:1 / 30대:2 / 40대:3 / 50대:4 / 60대:5 / 70대:6 / 80대:7 / 90대이상:8


In [ ]:
# 한국 나이 기준
REFERENCE_YEAR = 2006 + 1

df['age'] = REFERENCE_YEAR - df['birth_year']

bins = [-np.inf, 19, 29, 39, 49, 59, 69, 79, 89, np.inf]
labels = [0, 1, 2, 3, 4, 5, 6, 7, 8]

df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=False).astype('Int64')

# -1 삭제
df = df[df['age_group'] != -1]

# 필요없어진 컬럼 제거
df = df.drop(columns=['birth_year', 'age'])

In [ ]:
df['age_group'].value_counts().sort_index()

In [ ]:
df['age_group'].value_counts().sort_index().plot(kind='bar')

## 5. hit_days 계산

`hit_days = `

- **Fixed/Live**: 각 유형별 독립 계산
- **Total**: F + L 일별 합산 후 판정


In [ ]:
# fixed_hit_days
_f = df_f.rename(columns={'UserID': 'user_id'})

fixed_hit_days = (
  _f[_f['BetsF'] > 0]
  .groupby('user_id')['WinF']
  .apply(lambda s: (s > 0).sum())
  .rename('fixed_hit_days')
  .reset_index()
)

fixed_hit_days

In [ ]:
# live_hit_days
_l = df_l.rename(columns={'UserID': 'user_id'})

live_hit_days = (
  _l[_l['BetsL'] > 0]
  .groupby('user_id')['WinL']
  .apply(lambda s: (s > 0).sum())
  .rename('live_hit_days')
  .reset_index()
)

live_hit_days

In [ ]:
# dataFrame 병합
# 결측치 평균값 대체

_fl = (
  pd.merge(
    _f[['user_id', 'DateBet', 'BetsF', 'WinF']],
    _l[['user_id', 'DateBet', 'BetsL', 'WinL']],
    on=['user_id', 'DateBet'],
    how='outer',
  ).fillna(df.mean())
  .assign(
    Bets=lambda df: df['BetsF'] + df['BetsL'],
    Win=lambda df: df['WinF'] + df['WinL'],
  )
)

In [ ]:
# total_hit_days
total_hit_days = (
  _fl[_fl['Bets'] > 0]
  .groupby('user_id')['Win']
  .apply(lambda s: (s > 0).sum())
  .rename('total_hit_days')
  .reset_index()
)

fixed_hit_days

## 6. win_rate 계산

`win_rate = (Win > Stake) / (Bets > 0)`

- **Fixed/Live**: 각 유형별 독립 계산
- **Total**: F + L 일별 합산 후 판정


In [ ]:
# fixed_win_rate
fixed_win_rate = (
    _f[_f['BetsF'] > 0]
    .assign(_win=lambda df: df['WinF'] > df['StakeF'])
    .groupby('user_id')['_win']
    .mean()
    .rename('fixed_win_rate')
    .reset_index()
)

fixed_win_rate

In [ ]:
# live_win_rate
live_win_rate = (
    _l[_l['BetsL'] > 0]
    .assign(_win=lambda df: df['WinL'] > df['StakeL'])
    .groupby('user_id')['_win']
    .mean()
    .rename('live_win_rate')
    .reset_index()
)

live_win_rate

In [ ]:
# _fl에 Stake가 없으므로 Stake 포함하여 별도 merge
_fl_w = (
  pd.merge(
    _f[['user_id', 'DateBet', 'StakeF', 'WinF', 'BetsF']],
    _l[['user_id', 'DateBet', 'StakeL', 'WinL', 'BetsL']],
    on=['user_id', 'DateBet'],
    how='outer',
  )
  .assign(
    Stake=lambda df: df['StakeF'] + df['StakeL'],
    Win=lambda df: df['WinF'] + df['WinL'],
    Bets=lambda df: df['BetsF'] + df['BetsL'],
  ) 

)

In [ ]:
total_win_rate = (
    _fl_w[_fl_w['Bets'] > 0]
    .assign(_win=lambda x: x['Win'] > x['Stake'])
    .groupby('user_id')['_win']
    .mean()
    .rename('total_win_rate')
    .reset_index()
)

total_win_rate

## 7. avg_roi 계산

`avg_roi = mean((Win - Stake) / Stake)` where `Bets > 0 && Stake > 0`

- 일별 ROI를 먼저 계산한 뒤 유저별 평균 → 베팅 빈도 관계없이 하루하루의 수익률을 동등하게 반영
- **Total**: `_fl_w`(win_rate에서 생성, F + L 일별 합산) 재사용


In [ ]:
# fixed_avg_roi
# stake가 0 제외

fixed_avg_roi = (
  _f[_f['BetsF'] > 0]
  .loc[lambda df: df['StakeF'] > 0]
  .assign(_roi=lambda df: (df['WinF'] - df['StakeF']) / df['StakeF'])
  .groupby('user_id')['_roi']
  .mean()
  .rename('fixed_avg_roi')
  .reset_index()
)

fixed_avg_roi

In [ ]:
# live_avg_roi
live_avg_roi = (
  _l[_l['BetsL'] > 0]
  .loc[lambda df: df['StakeL'] > 0]
  .assign(_roi=lambda df: (df['WinL'] - df['StakeL']) / df['StakeL'])
  .groupby('user_id')['_roi']
  .mean()
  .rename('live_avg_roi')
  .reset_index()
)

live_avg_roi

In [ ]:
# total_avg_roi
total_avg_roi = (
    _fl_w[_fl_w['Bets'] > 0]
    # stakFe와 StakeL의 합이 0인 경우는 제외
    .loc[lambda df: df['Stake'] > 0]
    .assign(_roi=lambda x: (x['Win'] - x['Stake']) / x['Stake'])
    .groupby('user_id')['_roi']
    .mean()
    .rename('total_avg_roi')
    .reset_index()
)

total_avg_roi

## 8. churn 계산

기준일: 데이터 마지막 날짜 `2006-08-31`

- `0`: 기준일로부터 13개월(395일) 이내 베팅 활동 있음
- `1`: 없음


In [ ]:
REFERENCE_DATE = pd.Timestamp('2006-08-31')

last_bet = (
    pd.concat([
        _f.groupby('user_id')['DateBet'].max().rename('last_f'),
        _l.groupby('user_id')['DateBet'].max().rename('last_l'),
    ], axis=1)
    .max(axis=1)
    .rename('recency')
    .reset_index()
)

last_bet['churn'] = (
    (REFERENCE_DATE - last_bet['recency']).dt.days > 395
).astype(int)

last_bet['churn'].value_counts()

## 9. 최종 DataFrame 병합


In [ ]:
result = df.copy()

for aux in [
    fixed_hit_days,  live_hit_days,  total_hit_days,
    fixed_win_rate,  live_win_rate,  total_win_rate,
    fixed_avg_roi,   live_avg_roi,   total_avg_roi,
]:
    result = result.merge(aux, on='user_id', how='outer')

result = result.merge(last_bet[['user_id', 'churn']], on='user_id', how='outer')

FINAL_COLS = [
    'user_id', 'country_id', 'gender', 'age_group', 'reg_date', 'first_deposit',
    'first_bet',
    'fixed_bet_amount',  'live_bet_amount',  'total_bet_amount',
    'fixed_win_amount',  'live_win_amount',  'total_win_amount',
    'fixed_bet_cnt',     'live_bet_cnt',     'total_bet_cnt',
    'fixed_active_days', 'live_active_days', 'total_active_days',
    'fixed_hit_days',    'live_hit_days',    'total_hit_days',
    'fixed_win_rate',    'live_win_rate',    'total_win_rate',
    'fixed_avg_roi',     'live_avg_roi',     'total_avg_roi',
    'churn',
]

result = result[FINAL_COLS]

# 소수점 2자리로 반올림
float_cols = result.select_dtypes(include='float').columns.tolist()
result = result.assign(**{col: result[col].round(2) for col in float_cols})

result.shape


## 10. 저장


In [ ]:
OUTPUT_PATH = ROOT / 'data' / 'processed' / 'ljh_preprocessed.csv'
result.to_csv(OUTPUT_PATH, index=False)